This is an analysis of the *San Francisco Salaries* dataset acquired from the following link at [Kaggle](https://www.kaggle.com/datasets/kaggle/sf-salaries?resource=download).

THe data is for San Francisco city employees from 2011-2014. This allows for a comparisons in one broad category: how compensation is distributed and how it changed over the four-year period. This encompases a variety of aspects.

Compensation distribution can refer to the whole government budger and within specific groups or positions. For example, looking at what portion of a position's compensation is overtime, and what portion of the whole budget goes towards IT Workers in general.

In [1]:
import sqlite3 as sql
import pandas as pd

In [2]:
con = sql.connect("Salaries.sqlite")
cur = con.cursor()

In [3]:
query = "SELECT name FROM sqlite_master WHERE type='table';"
cur.execute(query)
tables = cur.fetchall()

In [4]:
columnDict = {}

for i,table in enumerate(tables):
    query = "SELECT * FROM %s;" % table
    cur.execute(query)
    cols = list(cur.description)
    valuelist = []
    for j, col in enumerate(cols):
        collist = list(col)
        valuelist.append(collist[0])
    columnDict[table] = valuelist

columnDict

{('Salaries',): ['Id',
  'EmployeeName',
  'JobTitle',
  'BasePay',
  'OvertimePay',
  'OtherPay',
  'Benefits',
  'TotalPay',
  'TotalPayBenefits',
  'Year',
  'Notes',
  'Agency',
  'Status']}

In [5]:
query = """ SELECT DISTINCT Year from Salaries"""
cur.execute(query)
cur.fetchall()

[(2011,), (2012,), (2013,), (2014,)]

In [6]:
query = """ SELECT Year, COUNT(Year) from Salaries GROUP By Year"""
cur.execute(query)
cur.fetchall()

[(2011, 36159), (2012, 36766), (2013, 37606), (2014, 38123)]

In [7]:
query = """ SELECT DISTINCT Notes from Salaries"""
cur.execute(query)
cur.fetchall()

[('',)]

In [8]:
query = """ SELECT DISTINCT Agency from Salaries"""
cur.execute(query)
cur.fetchall()

[('San Francisco',)]

In [9]:
query = """ SELECT DISTINCT Status from Salaries"""
cur.execute(query)
cur.fetchall()

[('',), ('PT',), ('FT',)]

In [10]:
query = """ SELECT Status, COUNT(Status) from Salaries GROUP By Status"""
cur.execute(query)
cur.fetchall()

[('', 110535), ('FT', 22334), ('PT', 15785)]

In [11]:
query = """SELECT DISTINCT JobTitle from Salaries"""
cur.execute(query)
cur.fetchmany(5)

[('GENERAL MANAGER-METROPOLITAN TRANSIT AUTHORITY',),
 ('CAPTAIN III (POLICE DEPARTMENT)',),
 ('WIRE ROPE CABLE MAINTENANCE MECHANIC',),
 ('DEPUTY CHIEF OF DEPARTMENT,(FIRE DEPARTMENT)',),
 ('ASSISTANT DEPUTY CHIEF II',)]

In [12]:
query = """SELECT COUNT(DISTINCT JobTitle) from Salaries"""
cur.execute(query)
cur.fetchmany(5)

[(2159,)]

In [13]:
query = """ SELECT DISTINCT Status from Salaries"""
cur.execute(query)
cur.fetchall()

[('',), ('PT',), ('FT',)]

In [14]:
query = """ SELECT MIN(BasePay), MAX(BasePay), AVG(BasePay) from Salaries"""
cur.execute(query)
cur.fetchall()

[(-166.01, 'Not Provided', 66053.72928807836)]

In [15]:
query="Select JobTitle,BasePay from Salaries where BasePay < 0"
cur.execute(query)
cur.fetchall()

[('Junior Clerk', -166.01),
 ('Junior Clerk', -121.63),
 ('Junior Clerk', -109.22),
 ('Junior Clerk', -106.6),
 ('Junior Clerk', -101.88),
 ('Junior Clerk', -93.14),
 ('Junior Clerk', -87.38),
 ('Junior Clerk', -75.67),
 ('Junior Clerk', -59.59),
 ('Junior Clerk', -30.58),
 ('Clerk', -9.5)]

In [16]:
query="Select Count(JobTitle) from Salaries where JobTitle in ('Junior Clerk')"
cur.execute(query)
cur.fetchall()

[(596,)]

In [17]:
query = """ SELECT MIN(OvertimePay), MAX(OvertimePay), AVG(OvertimePay) from Salaries where OvertimePay Not in ('Not Provided')"""
cur.execute(query)
cur.fetchall()

[(-0.01, 245131.88, 5066.059886444668)]

In [18]:
query="Select JobTitle,OvertimePay from Salaries where OvertimePay < 0"
cur.execute(query)
cur.fetchall()

[('Senior Eligibility Worker', -0.01)]

In [19]:
query = """ SELECT MIN(OtherPay), MAX(OtherPay), AVG(OtherPay) from Salaries where OtherPay Not in ('Not Provided')"""
cur.execute(query)
cur.fetchall()

[(-7058.59, 400184.25, 3648.767296804574)]

In [20]:
query="Select JobTitle,OtherPay from Salaries where OtherPay < 0"
cur.execute(query)
cur.fetchall()

[('IS Business Analyst-Principal', -7058.59),
 ('Custodial Supervisor', -9.6),
 ('Gardener', -46.76),
 ('Special Nurse', -50.19),
 ('Counselor, Log Cabin Ranch', -618.13)]

In [21]:
query = """ SELECT MIN(Benefits), MAX(Benefits), AVG(Benefits) from Salaries where Benefits Not in ('Not Provided')"""
cur.execute(query)
cur.fetchall()

[(-33.89, '', 18924.742068146654)]

In [22]:
query="Select JobTitle,Benefits from Salaries where Benefits < 0"
cur.execute(query)
cur.fetchall()

[('Police Officer 3', -2.73),
 ('Police Officer 3', -8.2),
 ('Police Officer 3', -33.89),
 ('Secretary 2', -13.8)]

In [23]:
query = """ SELECT MIN(TotalPay), MAX(TotalPay), AVG(TotalPay) from Salaries"""
cur.execute(query)
cur.fetchall()

[(-618.13, 567595.43, 74768.32197169265)]

In [24]:
query = """ SELECT MIN(TotalPayBenefits), MAX(TotalPayBenefits), AVG(TotalPayBenefits) from Salaries"""
cur.execute(query)
cur.fetchall()

[(-618.13, 567595.43, 93692.55481056681)]

# Data Exploration

With the broad strokes established, mainly what possible values there are for each category, the focus can be shifted on comparing and contrasting different groups of employees. Most of these are curiousities outside of a handful.

I'll be looking at the following:

* **Law Enforcement**: by looking at the strings *Police* and *Sheriff* I can see the Law Enforcement wages paid by the city.
* **Fire Department**: By taking a look at the string *Fire* I can see how much is spent on the Fire Department in terms of wages.
* **Mayor**: by querying 'Mayor' to take a look at the mayor and their staff.
* **Healthcare**: This one is a variety of strings that are cut off to bring in the largest variety of job titles. The list is *nurs* (gets nurse and nursery), *medical*, *health*, *dent*, *pharma* (pharmacy and pharmacist), *Physician*, and *Social Work*. This should get most healthcare workers though with so many strings it will probably get a few incorrect ones which is why I will also try to avoid the strings *resources* and *examiner*. 
* **Information Science**: All of the IT/IS positions start with 'IS' so looking for that with a space afterwards should get all of those positions without having to filter too many other job titles.
* **Justice**: *Defender* should pick up Public Defenders and *court* should pick up most who work at the court. Judge did not seem to get many results which is strange, and even looking for it directly did not give any results aside from a secretary.
* **Engineers and Scientists**: Looking at the strings *bio*, *chem*, *research* and *engineer* should get all the relevant job titles. This is to see how much money is spent on research, development, and engineers in general.
* **Planning**: Looking at the string *plann* should get all job titles related to urban planning.
* **Technicians and Mechanics**: *automotive*, *Electr*, *mech*, *plumb* should get all handymen, mechanics, and technicians. This is to see how much is spent on maintenance work and similar tasks.
* **Clerks**: *clerk* and *secretar* should bring up all clerks, secretaries and other clerical staff to see how much is spent on office staff.

In [71]:
query = """SELECT JobTitle, SUM(TotalPayBenefits), Year from Salaries WHERE JobTitle LIKE '%Judge%'"""
cur.execute(query)
cur.fetchall()

[('SECRETARY TO THE PRESIDING JUDGE', 2904666.62, 2011)]

## Law Enforcement Jobs

In [68]:
query = """SELECT JobTitle, SUM(TotalPayBenefits), Year from Salaries WHERE JobTitle LIKE '%Police%' OR '%Sheriff%' Group by JobTitle Order by Year"""
cur.execute(query)
cur.fetchall()

[('AIRPORT POLICE SERVICES AIDE', 11244297.15, 2011),
 ('ASSISTANT INSPECTOR (POLICE DEPARTMENT)', 188999.2, 2011),
 ('ASSISTANT INSPECTOR II (POLICE DEPARTMENT)', 1508888.61, 2011),
 ('ASSISTANT INSPECTOR III (POLICE DEPARTMENT)', 1374040.94, 2011),
 ('CAPTAIN III (POLICE DEPARTMENT)', 7836004.05, 2011),
 ('CHIEF OF POLICE', 267992.59, 2011),
 ('COMMANDER III, (POLICE DEPARTMENT)', 1435955.55, 2011),
 ('DEPUTY CHIEF III (POLICE DEPARTMENT)', 1250132.44, 2011),
 ('INSPECTOR II, (POLICE DEPARTMENT)', 432848.20999999996, 2011),
 ('INSPECTOR III, (POLICE DEPARTMENT)', 24968457.92, 2011),
 ('INSPECTOR, (POLICE DEPARTMENT)', 318997.97, 2011),
 ('INSTITUTIONAL POLICE LIEUTENANT', 59242.59, 2011),
 ('INSTITUTIONAL POLICE OFFICER', 945970.7899999999, 2011),
 ('INSTITUTIONAL POLICE SERGEANT', 253944.14, 2011),
 ('LIEUTENANT I, (POLICE DEPARTMENT)', 153795.73, 2011),
 ('LIEUTENANT II (POLICE DEPARTMENT)', 152985.39, 2011),
 ('LIEUTENANT III (POLICE DEPARTMENT)', 16364067.01, 2011),
 ('POLICE COM

In [73]:
query = """SELECT SUM(TotalPayBenefits), AVG(TotalPayBenefits), Count(Year), Year from Salaries WHERE JobTitle LIKE '%Police%' OR '%Sheriff%' Group by Year Order by Year"""
cur.execute(query)
cur.fetchall()

[(319268095.21, 127097.17166003183, 2512, 2011),
 (280530623.14, 146185.8380093799, 1919, 2012),
 (288728271.59, 154482.75633493846, 1869, 2013),
 (282011529.08, 148349.04212519724, 1901, 2014)]

## Fire Department Jobs

In [69]:
query = """SELECT JobTitle, SUM(TotalPayBenefits), Year from Salaries WHERE JobTitle LIKE '%Fire%' Group by JobTitle Order by Year"""
cur.execute(query)
cur.fetchall()

[('ASSISTANT CHIEF OF DEPARTMENT, (FIRE DEPARTMENT)', 610283.55, 2011),
 ('BATTALION CHIEF, (FIRE DEPARTMENT)', 9749499.16, 2011),
 ('CAPTAIN, BUREAU OF FIRE PREVENTION AND PUBLIC SAFE', 206704.63, 2011),
 ('CAPTAIN, FIRE SUPPRESSION', 12762877.81, 2011),
 ('CHIEF FIRE ALARM DISPATCHER', 112798.37, 2011),
 ('CHIEF OF DEPARTMENT, (FIRE DEPARTMENT)', 302377.73, 2011),
 ('DEPUTY CHIEF OF DEPARTMENT,(FIRE DEPARTMENT)', 838078.68, 2011),
 ('FIRE ALARM DISPATCHER', 130371.56999999999, 2011),
 ('FIRE FIGHTER PARAMEDIC', 38957981.84, 2011),
 ('FIRE PROTECTION ENGINEER', 507207.46, 2011),
 ('FIRE RESCUE PARAMEDIC', 566616.29, 2011),
 ('FIRE SAFETY INSPECTOR II', 1678561.17, 2011),
 ('FIREFIGHTER', 110597884.48, 2011),
 ('INSPECTOR, BUREAU OF FIRE PREVENTION AND PUBLIC SA', 3220814.25, 2011),
 ('INVESTIGATOR, BUREAU OF FIRE INVESTIGATION', 702051.63, 2011),
 ('LIEUTENANT, BUREAU OF FIRE PREVENTION AND PUBLIC S', 1534524.46, 2011),
 ('LIEUTENANT, FIRE DEPARTMENT', 27905707.74, 2011),
 ('MARINE EN

In [75]:
query = """SELECT SUM(TotalPayBenefits), AVG(TotalPayBenefits), Count(Year), Year from Salaries WHERE JobTitle LIKE '%Fire%' Group by Year Order by Year"""
cur.execute(query)
cur.fetchall()

[(211563265.94, 145005.66548320767, 1459, 2011),
 (273810371.21, 188965.05949620425, 1449, 2012),
 (291038363.49, 199341.3448561644, 1460, 2013),
 (285536817.28, 188972.0829119788, 1511, 2014)]

## Healthcare Jobs

In [79]:
query = """SELECT JobTitle, SUM(TotalPayBenefits), Year from Salaries WHERE JobTitle LIKE '%nurs%' OR '%medical%' OR '%dent%' OR '%pharma%' OR '%physician%' OR '%social work%' Group by JobTitle Order by Year"""
cur.execute(query)
cur.fetchall()

[('CHIEF NURSERY SPECIALIST', 84800.16, 2011),
 ('CLINICAL NURSE SPECIALIST', 3464581.28, 2011),
 ('LICENSED VOCATIONAL NURSE', 11591302.27, 2011),
 ('NURSE MANAGER', 13226873.2, 2011),
 ('NURSE MIDWIFE', 619265.27, 2011),
 ('NURSE PRACTITIONER', 18511994.55, 2011),
 ('NURSERY SPECIALIST', 343548.54000000004, 2011),
 ('NURSES STAFFING ASSISTANT', 786541.65, 2011),
 ('NURSING ASSISTANT', 8763814.85, 2011),
 ('NURSING SUPERVISOR', 4403712.41, 2011),
 ('NURSING SUPERVISOR PSYCHIATRIC', 866394.54, 2011),
 ('OPERATING ROOM NURSE', 224978.88, 2011),
 ('PUBLIC HEALTH NURSE', 5728069.39, 2011),
 ('REGISTERED NURSE', 129971170.5, 2011),
 ('SPECIAL NURSE', 43917756.47, 2011),
 ('Chief Nursery Specialist', 281776.61, 2012),
 ('Clinical Nurse Specialist', 13847920.48, 2012),
 ('Licensed Vocational Nurse', 55606050.12, 2012),
 ('Nurse Manager', 56737202.01, 2012),
 ('Nurse Midwife', 2427863.01, 2012),
 ('Nurse Practitioner', 84172842.49, 2012),
 ('Nursery Specialist', 1385957.88, 2012),
 ('Nurses S